# SpaceX Data Collection and Wrangling - Completed
This notebook combines the API collection, web scraping, and wrangling steps used in the IBM Applied Data Science Capstone.


## 1. Collect SpaceX launch data with REST API


In [ ]:
import requests, pandas as pd, numpy as np
api_url = "https://api.spacexdata.com/v4/launches/past"
launches = requests.get(api_url, timeout=30).json()
api_df = pd.json_normalize(launches)
api_df.head()


## 2. Enrich records using rocket, payload, launchpad and core endpoints


In [ ]:
def get_json(url):
    return requests.get(url, timeout=30).json()

rockets = pd.DataFrame(get_json("https://api.spacexdata.com/v4/rockets"))[["id","name"]]
payloads = pd.json_normalize(get_json("https://api.spacexdata.com/v4/payloads"))
launchpads = pd.DataFrame(get_json("https://api.spacexdata.com/v4/launchpads"))[["id","name","latitude","longitude"]]
cores = pd.DataFrame(get_json("https://api.spacexdata.com/v4/cores"))
rockets.head()


## 3. Web scrape Falcon 9 launch history


In [ ]:
import requests
from bs4 import BeautifulSoup
wiki_url = "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches"
html = requests.get(wiki_url, timeout=30).text
soup = BeautifulSoup(html, "html.parser")
tables = pd.read_html(html)
launch_tables = [t for t in tables if len(t.columns) >= 5]
len(launch_tables)


## 4. Wrangle the capstone analysis dataset


In [ ]:
data = pd.read_csv("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv")
if "PayloadMass" in data.columns:
    data["PayloadMass"] = data["PayloadMass"].fillna(data["PayloadMass"].mean())
landing_outcomes = {"True ASDS", "True RTLS", "True Ocean"}
if "Outcome" in data.columns:
    data["Class"] = data["Outcome"].apply(lambda x: 1 if str(x) in landing_outcomes else 0)
data.to_csv("dataset_part_2.csv", index=False)
data.head()


## Deliverable
The cleaned dataset contains Falcon 9 launch records and the binary Class target (1 = successful landing, 0 = unsuccessful/no landing).
